In [ ]:
!pip install langchain-huggingface -q

In [ ]:
# ─────────────────────────────────────────────
# Part 1. 라이브러리 설치 및 드라이브 마운트
# ─────────────────────────────────────────────
# 공식 faiss-gpu 대신 최신 환경과 호환되는 faiss-gpu-cu12를 설치합니다.
!pip install sentence-transformers langchain-huggingface langchain-community faiss-gpu-cu12

import json
import numpy as np
from google.colab import drive
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# 구글 드라이브 마운트 (팝업창에서 권한 허용 필요)
drive.mount('/content/drive')

print("라이브러리 임포트 및 드라이브 마운트 완료")

# ─────────────────────────────────────────────
# Part 2. 청크 파일 로드
# ─────────────────────────────────────────────

FILE_PATH = "/content/ingredient_chunks_preset3.json"

with open(FILE_PATH, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

print(f"총 청크 수: {len(all_chunks)}")

# ─────────────────────────────────────────────
# Part 3. LangChain Document 객체로 변환
# ─────────────────────────────────────────────
docs = [
    Document(
        page_content=chunk["page_content"],
        metadata=chunk["metadata"]
    )
    for chunk in all_chunks
]

# ─────────────────────────────────────────────
# Part 4. 임베딩 모델 로드 (GPU 할당)
# ─────────────────────────────────────────────
# device를 "cuda"로 설정하여 T4 GPU를 사용하도록 합니다.

embedding_model = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={"device": "cuda"},    # ★ 핵심: GPU 사용
    encode_kwargs={"normalize_embeddings": True},
)

print("임베딩 모델 로드 완료 (GPU)")

# 테스트 (768차원 확인)
test_vector = embedding_model.embed_query("나이아신아마이드 EWG 등급")
print(f"벡터 차원 수: {len(test_vector)}")


# ─────────────────────────────────────────────
# Part 5. FAISS 인덱스 구축
# ─────────────────────────────────────────────
# GPU를 사용하므로 로컬 CPU 대비 훨씬 빠르게 진행됩니다.

print("FAISS 인덱스 구축 시작... (GPU 가동 중 🚀)")

vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model,
)

print("FAISS 인덱스 구축 완료!")
print(f"저장된 벡터 수: {vectorstore.index.ntotal}")

# ─────────────────────────────────────────────
# Part 6. FAISS 인덱스 로컬(드라이브) 저장
# ─────────────────────────────────────────────
# 생성된 인덱스를 구글 드라이브에 저장하여 나중에 바로 불러올 수 있게 합니다.

FAISS_SAVE_PATH = "/content/drive/MyDrive/data/faiss_index_preset2"

vectorstore.save_local(FAISS_SAVE_PATH)
print(f"FAISS 인덱스 저장 완료: {FAISS_SAVE_PATH}")